In [0]:
USE CATALOG tibia_analytics;
USE SCHEMA utils;
SET TIME ZONE 'UTC';

In [0]:
INSERT OVERWRITE TABLE calendar
WITH base_dates AS (
  SELECT EXPLODE(
           SEQUENCE(
             TO_DATE('2026-02-10'), 
             TO_DATE('2100-12-31'), 
             INTERVAL 1 DAY
           )
         ) AS calendar_date
)
SELECT CAST(DATE_FORMAT(calendar_date, 'yyyyMMdd') AS INT)                                          AS date_key,
       YEAR(calendar_date)                                                                          AS year_key,
       YEAR(calendar_date) * 100 + QUARTER(calendar_date)                                           AS quarter_key,
       CAST(DATE_FORMAT(calendar_date, 'yyyyMM') AS INT)                                            AS month_key,
       YEAR(DATE_TRUNC('WEEK', calendar_date)) * 100 + WEEKOFYEAR(calendar_date)                    AS week_key,
       calendar_date                                                                                AS full_date,
       YEAR(calendar_date)                                                                          AS year,
       QUARTER(calendar_date)                                                                       AS quarter,
       CONCAT('Q', QUARTER(calendar_date))                                                          AS quarter_name,
       CONCAT(YEAR(calendar_date), '-Q', QUARTER(calendar_date))                                    AS quarter_year,
       CAST(DATE_TRUNC('QUARTER', calendar_date) AS DATE)                                           AS quarter_start_date,
       DATE_SUB(ADD_MONTHS(CAST(DATE_TRUNC('QUARTER', calendar_date) AS DATE), 3), 1)               AS quarter_end_date,
       MONTH(calendar_date)                                                                         AS month,
       DATE_FORMAT(calendar_date, 'MMMM')                                                           AS month_name,
       DATE_FORMAT(calendar_date, 'MMM')                                                            AS month_short_name,
       DATE_FORMAT(calendar_date, 'MMM/yy')                                                         AS month_year_short,
       CAST(DATE_TRUNC('MONTH', calendar_date) AS DATE)                                             AS month_start_date,
       LAST_DAY(calendar_date)                                                                      AS month_end_date,
       DAY(LAST_DAY(calendar_date))                                                                 AS days_in_month,
       WEEKOFYEAR(calendar_date)                                                                    AS iso_week,
       YEAR(DATE_TRUNC('WEEK', calendar_date))                                                      AS iso_year,
       YEAR(DATE_TRUNC('WEEK', calendar_date)) * 100 + WEEKOFYEAR(calendar_date)                    AS iso_year_week,
       CAST(DATE_TRUNC('WEEK', calendar_date) AS DATE)                                              AS week_start_date,
       DATE_ADD(CAST(DATE_TRUNC('WEEK', calendar_date) AS DATE), 6)                                 AS week_end_date,
       DAYOFMONTH(calendar_date)                                                                    AS day,
       DAYOFYEAR(calendar_date)                                                                     AS day_of_year,
       PMOD(DAYOFWEEK(calendar_date) + 5, 7) + 1                                                    AS day_of_week_iso,
       DATE_FORMAT(calendar_date, 'EEEE')                                                           AS day_name,
       DATE_FORMAT(calendar_date, 'EEE')                                                            AS day_short_name,
       DAYOFMONTH(calendar_date) = 1                                                                AS is_first_day_of_month,
       calendar_date = LAST_DAY(calendar_date)                                                      AS is_last_day_of_month,
       CASE WHEN DATE_FORMAT(calendar_date, 'E') NOT IN ('Sat', 'Sun') THEN TRUE ELSE FALSE END     AS is_weekday,
       CASE WHEN DATE_FORMAT(calendar_date, 'E')     IN ('Sat', 'Sun') THEN TRUE ELSE FALSE END     AS is_weekend,
       CURRENT_TIMESTAMP()                                                                          AS processed_at
  FROM base_dates